# Sycophancy Causal Effect — Analysis

This notebook performs the analysis of the inference results produced by `01_pilot.ipynb`.

**Decoupled from inference**: it reads the saved CSV from `results/`, so it can be run locally with a plain Python kernel — no GPU, no Colab, no model loading required.

**Sections**:
1. Setup and data loading
2. Aggregate statistics per (family, role, level)
3. Causal effect (ATE) per (family, level)
4. Statistical significance tests (paired t-tests + bootstrap CIs + Cohen's d_z)
5. Visualizations (forest plot, interaction plot, heatmap)
6. Per-question analysis (top extreme cases, scatter, distribution, per-category)
7. Save statistical tables

All figures are saved as PDF in `figures/` for direct import into the LaTeX paper.
All statistical tables are saved as CSV in `results/`.


## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve paths relative to the repo root (notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR  = PROJECT_ROOT / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Available result files:")
for f in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {f.name}")

In [ ]:
# Output directory for figures (PDF for LaTeX, high DPI)
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
print(f"Figures dir: {FIGURES_DIR}")

In [ ]:
# Load the main experiment results
RESULTS_FILE = RESULTS_DIR / "main_multifamily_n300.csv"
df = pd.read_csv(RESULTS_FILE)

print(f"Loaded {len(df)} records from {RESULTS_FILE.name}")
print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFamilies: {sorted(df['family'].unique())}")
print(f"Roles:    {sorted(df['role'].unique())}")
print(f"Levels:   {sorted(df['level'].unique())}")
print(f"\nFirst few rows:")
df.head()

## 2. Aggregate Statistics

Mean and standard deviation of `p_agree` for each combination of (family, role, level).
The `count` column verifies that all cells contain the expected number of examples (n=300).

In [ ]:
agg = (
    df.groupby(["family", "role", "level"])["p_agree"]
      .agg(["mean", "std", "count"])
      .round(3)
)
print("=== Mean P(agree) per (family, role, level) ===")
print(agg)

## 3. Treatment Effect (ATE)

The Average Treatment Effect (ATE) of instruction tuning at each premise strength,
within each model family:

$$\text{ATE}_{f, \ell} = \bar{P}_\text{agree}(\text{instruct}, f, \ell) - \bar{P}_\text{agree}(\text{base}, f, \ell)$$

A positive ATE means the instruct model agrees more with the user's claim than the base model — i.e., higher sycophancy. A negative ATE means the instruct model disagrees more — i.e., higher resistance to user-stated falsehoods.

In [ ]:
pivot = (
    df.groupby(["family", "level", "role"])["p_agree"]
      .mean()
      .unstack("role")
)
pivot["ATE"] = pivot["instruct"] - pivot["base"]

print("=== Treatment effect (instruct - base) per (family, level) ===")
print(pivot.round(3))

## 4. Statistical Significance

Paired t-tests on the per-example differences `(p_agree_instruct - p_agree_base)` to determine whether the observed ATEs are statistically significant.

For each (family, level) cell we compute:
- **paired t-test** on the per-example differences
- **non-parametric bootstrap** of the mean difference (95% CI, 10,000 resamples)
- **Cohen's d_z** as standardized effect size for paired designs

Bonferroni correction for 12 simultaneous tests: α' = 0.05/12 ≈ 0.0042.

In [ ]:
from scipy import stats


def compute_paired_test(df, family, level, n_bootstrap=10000, seed=42):
    """
    Compute paired t-test and bootstrap CI for the ATE within a (family, level) cell.

    Pairs observations by example_id, computes (p_agree_instruct - p_agree_base)
    per example, then runs:
      - paired t-test on the per-example differences
      - non-parametric bootstrap of the mean difference (95% CI)
      - Cohen's d_z (standardized effect size for paired designs)

    Returns:
        dict with ATE, t_stat, p_value, ci_low_95, ci_high_95, cohen_dz, n.
    """
    sub = df[(df["family"] == family) & (df["level"] == level)]
    pivot = sub.pivot(index="example_id", columns="role", values="p_agree")
    pivot = pivot.dropna(subset=["base", "instruct"])
    diff = (pivot["instruct"] - pivot["base"]).values

    n = len(diff)
    ate = diff.mean()
    sd = diff.std(ddof=1)

    # Paired t-test (equivalent to one-sample t-test on the differences)
    t_stat, p_value = stats.ttest_rel(pivot["instruct"], pivot["base"])

    # Bootstrap 95% CI on the mean difference
    rng = np.random.default_rng(seed)
    boot_means = np.array([
        rng.choice(diff, size=n, replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

    # Cohen's d_z for paired data: mean / sd of the differences
    cohen_dz = ate / sd if sd > 0 else np.nan

    return {
        "family":    family,
        "level":     level,
        "n":         n,
        "ATE":       ate,
        "t_stat":    t_stat,
        "p_value":   p_value,
        "ci_low_95": ci_low,
        "ci_high_95": ci_high,
        "cohen_dz":  cohen_dz,
    }


# Run the test for every (family, level) cell
families = sorted(df["family"].unique())
levels   = ["L0_neutral", "L1_weak", "L2_medium", "L3_strong"]

results = [compute_paired_test(df, f, l) for f in families for l in levels]
stats_df = pd.DataFrame(results)

# Format for readable output
display_cols = ["family", "level", "n", "ATE", "ci_low_95", "ci_high_95", "t_stat", "p_value", "cohen_dz"]
print("=== Statistical tests of ATE per (family, level) ===\n")
print(stats_df[display_cols].round(4).to_string(index=False))

## 5. Visualizations

Three core figures for the paper:
- **Forest plot**: ATE with 95% CIs, with markers indicating Bonferroni-significant cells
- **Interaction plot**: P(agree) vs premise strength, separated by family and role
- **Heatmap**: ATE matrix across families and levels, color-coded

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Apply a clean style
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

# Consistent ordering across plots
FAMILY_ORDER = ["Qwen2.5-1.5B", "Llama-3.2-1B", "Gemma-2-2B"]
LEVEL_ORDER  = ["L0_neutral", "L1_weak", "L2_medium", "L3_strong"]
LEVEL_LABEL  = {"L0_neutral": "L0\nneutral", "L1_weak": "L1\nweak",
                "L2_medium": "L2\nmedium", "L3_strong": "L3\nstrong"}
FAMILY_COLORS = {"Qwen2.5-1.5B": "#2E86AB", "Llama-3.2-1B": "#A23B72", "Gemma-2-2B": "#F18F01"}

### 5.1 Forest plot of ATEs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

y_pos = 0
yticks, ylabels = [], []

for family in FAMILY_ORDER:
    for level in LEVEL_ORDER:
        row = stats_df[(stats_df["family"] == family) & (stats_df["level"] == level)].iloc[0]
        ate, lo, hi, p = row["ATE"], row["ci_low_95"], row["ci_high_95"], row["p_value"]

        # Marker filled if significant after Bonferroni (p < 0.0042), open otherwise
        is_sig = p < 0.05 / 12
        marker_face = FAMILY_COLORS[family] if is_sig else "white"

        ax.errorbar(ate, y_pos, xerr=[[ate - lo], [hi - ate]],
                    fmt="o", color=FAMILY_COLORS[family], mfc=marker_face,
                    markersize=8, capsize=4, lw=1.5)
        yticks.append(y_pos)
        ylabels.append(f"{family}  {level.split('_')[1]}")
        y_pos += 1
    y_pos += 0.5  # gap between families

ax.axvline(0, color="black", lw=0.8, linestyle="--")
ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=9)
ax.set_xlabel("ATE = mean P(agree | instruct) − mean P(agree | base)")
ax.set_title("Forest plot: causal effect of instruction tuning on sycophancy\n"
             "(filled markers significant after Bonferroni correction)", fontsize=11)
ax.invert_yaxis()
plt.tight_layout()

plt.savefig(FIGURES_DIR / "fig1_forest_ATE.pdf", dpi=300, bbox_inches="tight")
plt.show()

### 5.2 Interaction plot: P(agree) by family and role

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

agg_means = (
    df.groupby(["family", "level", "role"])["p_agree"]
      .agg(["mean", "std", "count"])
      .reset_index()
)
agg_means["sem"] = agg_means["std"] / np.sqrt(agg_means["count"])

for ax, family in zip(axes, FAMILY_ORDER):
    sub = agg_means[agg_means["family"] == family]

    for role, marker in [("base", "o"), ("instruct", "s")]:
        s = sub[sub["role"] == role].set_index("level").loc[LEVEL_ORDER]
        ax.errorbar(range(len(LEVEL_ORDER)), s["mean"], yerr=1.96 * s["sem"],
                    marker=marker, label=role, capsize=3, lw=2, markersize=7)

    ax.set_xticks(range(len(LEVEL_ORDER)))
    ax.set_xticklabels([LEVEL_LABEL[l] for l in LEVEL_ORDER])
    ax.axhline(0.5, color="grey", lw=0.6, linestyle=":")
    ax.set_title(family)
    ax.set_ylim(0, 1)
    if ax is axes[0]:
        ax.set_ylabel("P(agree)")
    ax.legend(loc="upper left", frameon=False)

fig.suptitle("Interaction plot: P(agree) by premise strength, split by family and role",
             fontsize=11, y=1.02)
plt.tight_layout()

plt.savefig(FIGURES_DIR / "fig2_interaction_pagree.pdf", dpi=300, bbox_inches="tight")
plt.show()

### 5.3 Heatmap of ATEs

In [ ]:
ate_matrix = (
    stats_df.pivot(index="family", columns="level", values="ATE")
            .reindex(FAMILY_ORDER)
            [LEVEL_ORDER]
)

fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(ate_matrix, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            cbar_kws={"label": "ATE"}, ax=ax, vmin=-0.25, vmax=0.25,
            linewidths=0.5)
ax.set_title("ATE heatmap: instruction tuning effect on P(agree)", fontsize=11)
ax.set_xlabel("Premise strength")
ax.set_ylabel("Model family")
plt.tight_layout()

plt.savefig(FIGURES_DIR / "fig3_heatmap_ATE.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 6. Per-Question Analysis

Beyond aggregate statistics, we drill down to individual questions to understand **which** prompts drive the observed effects, and **how** the effect distribution looks per family.

This section covers:
- 6.1 Computing per-example differences (instruct − base)
- 6.2 Top-N most sycophantic and most resistant questions per family
- 6.3 Scatter plot of per-question agreement at L3
- 6.4 Distribution of per-question differences across levels
- 6.5 Aggregation by TruthfulQA category

### 6.1 Per-example differences

In [ ]:
# Pivot to get base and instruct as separate columns, per (example, family, level)
per_example = (
    df.pivot_table(
        index=["family", "level", "example_id", "question", "wrong_answer"],
        columns="role",
        values="p_agree",
    )
    .reset_index()
)
per_example["diff"] = per_example["instruct"] - per_example["base"]

print(f"Shape: {per_example.shape}")
print(f"Each row = one example × family × level (3 families × 4 levels × 300 examples = {3*4*300} rows)")
per_example.head()

### 6.2 Top extreme questions at L3

In [ ]:
def show_extremes(per_example, family, level, n=5):
    """Print the n most sycophantic and n most resistant examples for a (family, level) cell."""
    sub = (per_example[(per_example["family"] == family) & 
                       (per_example["level"] == level)]
           .sort_values("diff", ascending=False))
    
    print(f"\n{'='*70}")
    print(f"  {family}  /  {level}")
    print(f"{'='*70}")
    
    print(f"\n  TOP {n} MOST SYCOPHANTIC (instruct agrees more with the false claim)")
    print(f"  {'-'*66}")
    for _, row in sub.head(n).iterrows():
        print(f"  diff = {row['diff']:+.3f}    "
              f"P(base)={row['base']:.2f}    P(instruct)={row['instruct']:.2f}")
        print(f"    Q: {row['question']}")
        print(f"    False claim: {row['wrong_answer']}")
        print()
    
    print(f"\n  TOP {n} MOST RESISTANT (instruct disagrees more with the false claim)")
    print(f"  {'-'*66}")
    for _, row in sub.tail(n)[::-1].iterrows():
        print(f"  diff = {row['diff']:+.3f}    "
              f"P(base)={row['base']:.2f}    P(instruct)={row['instruct']:.2f}")
        print(f"    Q: {row['question']}")
        print(f"    False claim: {row['wrong_answer']}")
        print()


# Inspect the strongest level (L3) for each family
for family in FAMILY_ORDER:
    show_extremes(per_example, family, "L3_strong", n=5)

### 6.3 Scatter plot at L3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharex=True, sharey=True)

for ax, family in zip(axes, FAMILY_ORDER):
    sub = per_example[(per_example["family"] == family) & 
                      (per_example["level"] == "L3_strong")]
    
    # Diagonal reference: y = x
    ax.plot([0, 1], [0, 1], "k--", lw=0.6, alpha=0.4, zorder=1)
    
    # Scatter of individual questions
    ax.scatter(sub["base"], sub["instruct"], 
               alpha=0.5, s=25, c=FAMILY_COLORS[family],
               edgecolors="white", linewidths=0.4, zorder=2)
    
    # Mean point (red X)
    ax.scatter([sub["base"].mean()], [sub["instruct"].mean()], 
               s=200, c="red", marker="X", linewidths=2,
               edgecolors="black", zorder=3, label="mean")
    
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.set_aspect("equal")
    ax.set_title(family, fontsize=11)
    ax.set_xlabel("P(agree | base)")
    ax.legend(loc="upper left", frameon=False)

axes[0].set_ylabel("P(agree | instruct)")
fig.suptitle("Per-question agreement at L3_strong: each point is one question", 
             fontsize=11, y=1.02)
plt.tight_layout()

plt.savefig(FIGURES_DIR / "fig4_scatter_perexample_L3.pdf", dpi=300, bbox_inches="tight")
plt.show()

### 6.4 Distribution of per-example differences

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharex=True, sharey=True)

for ax, level in zip(axes, LEVEL_ORDER):
    for family in FAMILY_ORDER:
        sub = per_example[(per_example["family"] == family) & 
                          (per_example["level"] == level)]
        ax.hist(sub["diff"], bins=25, alpha=0.55, label=family, color=FAMILY_COLORS[family])
    
    ax.axvline(0, color="black", lw=0.7, linestyle="--", alpha=0.7)
    ax.set_title(level, fontsize=10)
    ax.set_xlabel("p_agree(instruct) − p_agree(base)")

axes[0].set_ylabel("Number of questions")
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Per-question distribution of the instruction-tuning effect, by level", 
             fontsize=11, y=1.02)
plt.tight_layout()

plt.savefig(FIGURES_DIR / "fig5_histogram_diff_perlevel.pdf", dpi=300, bbox_inches="tight")
plt.show()

### 6.5 Per-category analysis

To understand **what kinds of questions** drive the family-specific patterns, we aggregate the per-example differences by TruthfulQA category and family at L3_strong (the strongest premise level).

We re-load TruthfulQA's `generation` config to recover the `category` field and merge it with our per-example data.

In [ ]:
from datasets import load_dataset

# Load TruthfulQA generation config and extract (question -> category) mapping
ds_truthful = load_dataset("truthful_qa", "generation")
truthful_meta = pd.DataFrame({
    "question": ds_truthful["validation"]["question"],
    "category": ds_truthful["validation"]["category"],
})

print(f"Loaded {len(truthful_meta)} questions with category metadata")
print(f"Unique categories in TruthfulQA: {truthful_meta['category'].nunique()}\n")
print("Most frequent categories in full TruthfulQA:")
print(truthful_meta["category"].value_counts().head(10))

In [ ]:
# Merge category info into per_example by question text
per_example_cat = per_example.merge(truthful_meta, on="question", how="left")

# Sanity check
n_unmatched = per_example_cat["category"].isna().sum()
print(f"Rows without category match: {n_unmatched} / {len(per_example_cat)}")
print(f"(should be 0 if all our questions were drawn from TruthfulQA)\n")

# Show how many questions per category in our 300-example sample
print("Categories represented in our 300-question sample:")
sample_cats = per_example_cat[
    (per_example_cat["family"] == "Qwen2.5-1.5B") & 
    (per_example_cat["level"] == "L3_strong")
]["category"].value_counts()
print(sample_cats.head(20))

In [ ]:
# Aggregate per (family, category) at L3_strong
cat_stats = (
    per_example_cat[per_example_cat["level"] == "L3_strong"]
    .groupby(["family", "category"])
    .agg(
        n=("diff", "size"),
        mean_diff=("diff", "mean"),
        std_diff=("diff", "std"),
    )
    .reset_index()
)

# Keep only categories with at least 5 questions in our sample (more stable estimates)
MIN_N = 5
cat_stats_filt = cat_stats[cat_stats["n"] >= MIN_N]
print(f"Categories with >= {MIN_N} questions: {cat_stats_filt['category'].nunique()}")

# Pivot to (category, family) heatmap, sorted by Qwen's ATE for visual narrative
heatmap_data = (
    cat_stats_filt
    .pivot(index="category", columns="family", values="mean_diff")
    [FAMILY_ORDER]
    .sort_values("Qwen2.5-1.5B")
)

print(f"\n=== Mean ATE by (category, family) at L3_strong ===")
print(heatmap_data.round(3))

In [ ]:
# Heatmap visualization
fig, ax = plt.subplots(figsize=(7, max(5, len(heatmap_data) * 0.35)))

sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            cbar_kws={"label": "ATE at L3_strong"}, ax=ax,
            vmin=-0.4, vmax=0.4, linewidths=0.5, linecolor="white")

ax.set_title("ATE by category and family at L3_strong\n"
             f"(categories with ≥ {MIN_N} questions; sorted by Qwen ATE)", fontsize=11)
ax.set_xlabel("Model family")
ax.set_ylabel("TruthfulQA category")
plt.tight_layout()

plt.savefig(FIGURES_DIR / "fig6_heatmap_ATE_by_category.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print("=" * 78)
print("  MOST SYCOPHANTIC CATEGORIES (highest +ATE) PER FAMILY @ L3_strong")
print("=" * 78)

for family in FAMILY_ORDER:
    sub = (cat_stats_filt[cat_stats_filt["family"] == family]
           .sort_values("mean_diff", ascending=False))
    print(f"\n  {family}:")
    for _, row in sub.head(5).iterrows():
        print(f"    {row['category']:32s}  ATE = {row['mean_diff']:+.3f}   (n={row['n']:>3})")

print("\n")
print("=" * 78)
print("  MOST RESISTANT CATEGORIES (lowest -ATE) PER FAMILY @ L3_strong")
print("=" * 78)

for family in FAMILY_ORDER:
    sub = (cat_stats_filt[cat_stats_filt["family"] == family]
           .sort_values("mean_diff", ascending=True))
    print(f"\n  {family}:")
    for _, row in sub.head(5).iterrows():
        print(f"    {row['category']:32s}  ATE = {row['mean_diff']:+.3f}   (n={row['n']:>3})")

## 7. Save Statistical Tables

Export the statistical tables as CSV in `results/` for direct citation in the paper and as appendix material.

In [ ]:
# Per-(family, level) statistics: ATE, CI, p-values, Cohen's d_z
stats_df_path = RESULTS_DIR / "stats_per_family_level.csv"
stats_df.to_csv(stats_df_path, index=False)

# Per-category breakdown at L3
cat_stats_path = RESULTS_DIR / "stats_per_category_family_L3.csv"
cat_stats_filt.to_csv(cat_stats_path, index=False)

# Aggregate means table (mean p_agree per condition)
agg_path = RESULTS_DIR / "agg_mean_pagree.csv"
agg.to_csv(agg_path)

print(f"Saved:")
print(f"  {stats_df_path}")
print(f"  {cat_stats_path}")
print(f"  {agg_path}")

## Summary of Findings

**Aggregate effects (Section 3-5):**

- **Qwen2.5-1.5B-Instruct** shows the *classical* sycophancy pattern: more resistant to weak/medium false premises (ATE ≈ −0.22) but cedes substantially under strong premises (ATE = +0.24). Effect sizes are extreme (|Cohen's d_z| up to 4.7).
- **Llama-3.2-1B-Instruct** shows the *opposite* pattern: stronger resistance than the base under strong premises (ATE = −0.20, d_z = −2.10), and a modest accuracy degradation at L0.
- **Gemma-2-2B-it** shows modest mean effects but **per-question polarization**: at L3, the distribution of differences is bimodal with peaks near ±0.4.

**Statistical significance (Section 4):** 11 of 12 (family × level) ATEs are significant at α = 0.05; 8 of them survive Bonferroni correction (α' = 0.0042).

**Per-question patterns (Section 6.2):** Qwen yields to *urban myths, outdated facts, and conspiracy-adjacent claims*; Llama resists most strongly on *stereotypes and historical facts*; Gemma's most-sycophantic cases concern *jurisdiction-specific legal claims and absolute superlatives*.

**Per-category patterns (Section 6.5):** Stereotypes function as a universal "safety wall" (all families more resistant than baseline). *Confusion* and *Logical Falsehood* categories are universal weak spots, with Gemma reaching ATE up to +0.51. *History* and *Religion* show maximum disagreement between Qwen and Llama (gap ≈ 0.6).

**Key takeaway:** the causal effect of instruction tuning on sycophancy is **family-dependent** in both magnitude and *direction*. Conclusions drawn from single-family studies may not generalize.